In [1]:
import os
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import psutil
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.area_size_shape_utils import (
    measure_3D_area_size_shape,
)
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)

# bug in the cucim module but we are using CPU so it does not matter for now
# from image_analysis_3D.featurization_utils.area_size_shape_utils_gpu import measure_3D_area_size_shape_gpu
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    get_mem_and_time_profiling,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    compartment = arguments_dict["compartment"]
    channel = arguments_dict["channel"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "C4-2"
    patient = "NF0014_T1"
    compartment = "Nuclei"
    channel = "DNA"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)

In [3]:
channel_n_compartment_mapping = {
    "DNA": "405",
    "AGP": "488",
    "ER": "555",
    "Mito": "640",
    "BF": "TRANS",
    "Nuclei": "nuclei_",
    "Cell": "cell_",
    "Cytoplasm": "cytoplasm_",
    "Organoid": "organoid_",
}

In [4]:
start_time = time.time()
# get starting memory (cpu)
start_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
)

In [6]:
object_loader = ObjectLoader(
    image=None,
    label_image=image_set_loader.image_set_dict[compartment],
    channel_name=None,
    compartment_name=compartment,
)

# area, size, shape
if processor_type == "GPU":
    size_shape_dict = measure_3D_area_size_shape_gpu(
        image_set_loader=image_set_loader,
        object_loader=object_loader,
    )
elif processor_type == "CPU":
    size_shape_dict = measure_3D_area_size_shape(
        image_set_loader=image_set_loader,
        object_loader=object_loader,
    )
else:
    raise ValueError(
        f"Processor type {processor_type} is not supported. Use 'CPU' or 'GPU'."
    )

In [7]:
final_df = pd.DataFrame(size_shape_dict)

# prepend compartment and channel to column names
for col in final_df.columns:
    if col not in ["object_id"]:
        final_df[col] = final_df[col].astype(np.float32)
        final_df.rename(
            columns={
                col: format_morphology_feature_name(
                    compartment=compartment,
                    channel=channel,
                    feature_type="AreaSizeShape",
                    measurement=col,
                )
            },
            inplace=True,
        )

final_df.insert(1, "image_set", image_set_loader.image_set_name)

output_file = pathlib.Path(
    output_parent_path
    / f"AreaSizeShape_{compartment}_{processor_type}_features.parquet"
)
final_df.to_parquet(output_file, index=False)
final_df.head()

,object_id,image_set,Nuclei_DNA_AreaSizeShape_Volume,Nuclei_DNA_AreaSizeShape_CenterX,Nuclei_DNA_AreaSizeShape_CenterY,Nuclei_DNA_AreaSizeShape_CenterZ,Nuclei_DNA_AreaSizeShape_BboxVolume,Nuclei_DNA_AreaSizeShape_MinX,Nuclei_DNA_AreaSizeShape_MaxX,Nuclei_DNA_AreaSizeShape_MinY,Nuclei_DNA_AreaSizeShape_MaxY,Nuclei_DNA_AreaSizeShape_MinZ,Nuclei_DNA_AreaSizeShape_MaxZ,Nuclei_DNA_AreaSizeShape_Extent,Nuclei_DNA_AreaSizeShape_EulerNumber,Nuclei_DNA_AreaSizeShape_EquivalentDiameter,Nuclei_DNA_AreaSizeShape_SurfaceArea
0,257,C4-2,88391.0,505.976227,557.246887,3.966252,125712.0,456.0,553.0,486.0,630.0,0.0,9.0,0.703123,1.0,55.267498,284.321991
1,1028,C4-2,88961.0,565.656860,803.809631,5.975922,156948.0,505.0,628.0,747.0,863.0,1.0,12.0,0.566818,1.0,55.386044,390.520294
2,1799,C4-2,89463.0,742.796631,386.613556,5.330461,129720.0,671.0,812.0,342.0,434.0,1.0,11.0,0.689662,1.0,55.490028,334.741425
3,2056,C4-2,89112.0,468.489960,469.192596,6.000045,154440.0,409.0,526.0,410.0,530.0,1.0,12.0,0.577001,1.0,55.417362,351.878937
4,2313,C4-2,77786.0,648.099609,591.448975,6.315738,147000.0,586.0,711.0,543.0,641.0,1.0,13.0,0.529156,1.0,52.962395,428.843231


In [8]:
end_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2
end_time = time.time()
get_mem_and_time_profiling(
    start_mem=start_mem,
    end_mem=end_mem,
    start_time=start_time,
    end_time=end_time,
    feature_type="AreaSizeShape",
    well_fov=well_fov,
    patient_id=patient,
    channel="NoChannel",
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{image_base_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_AreaSizeShape_DNA_{compartment}_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: C4-2
        Feature type: AreaSizeShape
        CPU/GPU: CPU
        Memory usage: 1817.94 MB
        Time elapsed:
        --- 24.23 seconds ---
        --- 0.40 minutes ---
        --- 0.01 hours ---
    


True